In [0]:
%sql
CREATE TEMPORARY VIEW vw_insight_proccesing_pipeline AS
WITH details_processing_rn as (
    SELECT 
        origin.pipeline_name,
        origin.flow_name,
        details,
        timestamp,
        row_number() OVER (PARTITION BY origin.flow_name, CAST(timestamp as date) ORDER BY timestamp DESC) as rn --select *
    FROM fintech_finpay.observability.event_log_finpay_etl
    WHERE event_type = 'flow_progress'
        AND details:flow_progress:status= 'RUNNING'
        AND origin.pipeline_name = 'finpay_etl_pipeline'
        AND origin.flow_name like 'fintech_finpay.silver.%'
        AND origin.flow_name <> 'fintech_finpay.silver.quarantine'
        AND message like 'Completed %'
)

SELECT
    CAST(timestamp as date) as processing_date,
    pipeline_name,
    trim(trailing ']' from substring_index(details:flow_progress:metrics:source_metrics[0]:source_name, '[', -1)) as source_table,
    split(flow_name, '\\.', 3)[1] as layer,
    flow_name as sink_table,
    CASE
        WHEN flow_name like 'fintech_finpay.silver.%' THEN details:flow_progress:metrics.num_upserted_rows
        ELSE details:flow_progress:metrics.num_output_rows
    END as upserted_rows,
    details:flow_progress:data_quality.warned_records as rejected_rows
FROM details_processing_rn
WHERE rn = 1
;

In [0]:
%sql

WITH details_processing_silver_rn as (
    SELECT 
        origin.pipeline_name,
        origin.flow_name,
        details,
        timestamp,
        row_number() OVER (PARTITION BY origin.flow_name, CAST(timestamp as date) ORDER BY timestamp DESC) as rn --select *
    FROM fintech_finpay.observability.event_log_finpay_etl
    WHERE event_type = 'flow_progress'
        AND details:flow_progress:status= 'RUNNING'
        AND origin.pipeline_name = 'finpay_etl_pipeline'
        AND origin.flow_name like 'fintech_finpay.silver.%'
        AND origin.flow_name <> 'fintech_finpay.silver.quarantine'
        AND message like 'Completed %'
),
explode_expectations as (
    SELECT
        CAST(timestamp as date) as processing_date,
        pipeline_name,
        trim(trailing ']' from substring_index(details:flow_progress:metrics:source_metrics[0]:source_name, '[', -1)) as source_table,
        split(flow_name, '\\.', 3)[1] as layer,
        flow_name as sink_table,
        explode(from_json(get_json_object(details, '$.flow_progress.data_quality.expectations'), 'ARRAY<STRUCT<name: STRING, failed_records: INT>>')) as expectations,
        CAST(details:flow_progress:data_quality.warned_records as INT) as total_rejected
    FROM details_processing_silver_rn
    WHERE rn = 1
)
SELECT
    processing_date,
    pipeline_name,
    layer,
    sink_table,
    expectations.name as expectation,
    CAST(expectations.failed_records as INT) as rejected_records,
    CAST((CAST(expectations.failed_records as INT) / total_rejected) as decimal(3,2)) as rate_rejected
FROM explode_expectations


In [0]:
%sql

SELECT *
FROM vw_insight_proccesing_pipeline
WHERE sink_table like '%.silver.transactions'
ORDER BY processing_date desc

In [0]:
%sql

SELECT *
FROM vw_insight_proccesing_pipeline
WHERE sink_table like '%.silver.users'
ORDER BY processing_date desc

In [0]:
%sql

SELECT *
FROM vw_insight_proccesing_pipeline
WHERE sink_table like '%.silver.merchants'
ORDER BY processing_date desc